# 5.5 Polynomial Regression

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 5: Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-05-linear-regression/05.05_polynomial-regression.ipynb)

Companion code for Section 5.5 of the book. Run the cells in order from top to bottom.

**Example 5.5.1** · bash

```bash
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# ── Generate non-linear data ──────────────────────────────────────
np.random.seed(42)
n = 100
X = np.sort(np.random.uniform(-3, 3, n)).reshape(-1, 1)
y = 0.5*X.squeeze()**3 - 1.5*X.squeeze() + 2 + np.random.normal(0, 1.5, n)
# True relationship: y = 0.5x³ - 1.5x + 2 + noise (cubic)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# ── Fit models of different degrees ──────────────────────────────
degrees   = [1, 2, 3, 5, 10, 15]
results   = {}
x_plot    = np.linspace(-3.2, 3.2, 300).reshape(-1,1)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Polynomial Regression - Effect of Degree', fontsize=14, fontweight='bold')

for idx, degree in enumerate(degrees):
    # Build Pipeline: PolynomialFeatures → Scaler → LinearRegression
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('lr',     LinearRegression())
    ])
    pipe.fit(X_train, y_train)

    y_pred_train = pipe.predict(X_train)
    y_pred_test  = pipe.predict(X_test)
    train_rmse   = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse    = np.sqrt(mean_squared_error(y_test, y_pred_test))
    train_r2     = pipe.score(X_train, y_train)
    test_r2      = pipe.score(X_test, y_test)
    results[degree] = {'train_rmse': train_rmse, 'test_rmse': test_rmse,
                       'train_r2': train_r2, 'test_r2': test_r2}

    # Plot
    row, col = idx // 3, idx % 3
    ax = axes[row, col]
    ax.scatter(X_train, y_train, color='#1976D2', s=20, alpha=0.6, label='Train')
    ax.scatter(X_test,  y_test,  color='#E53935', s=20, alpha=0.6, label='Test')
    ax.plot(x_plot, pipe.predict(x_plot), 'k-', linewidth=2)
    ax.set_ylim(-15, 15); ax.set_xlim(-3.5, 3.5)
    ax.set_title(f'Degree {degree}\nTrain RMSE={train_rmse:.2f}  Test RMSE={test_rmse:.2f}',
               fontsize=10)
    if degree in [1, 15]:
        ax.set_xlabel(f'{'← Underfitting' if degree==1 else 'Overfitting →'}', fontsize=10,
                     color='red', style='italic')
    elif degree == 3:
        ax.set_xlabel('← Optimal Fit →', fontsize=10, color='green', style='italic')
    if col == 0: ax.set_ylabel('y')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('polynomial_regression_degrees.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Print results summary ─────────────────────────────────────────
print(f'{'Degree':>8} {'Train R²':>10} {'Test R²':>10} {'Train RMSE':>12} {'Test RMSE':>12} {'Verdict'}')
print('-'*70)
for deg, r in results.items():
    verdict = 'UNDERFITTING' if r['test_r2'] < 0.7 and deg < 3 else (
              'OPTIMAL' if 2 <= deg <= 4 else (
              'OVERFITTING' if r['test_rmse'] > 3 and r['train_rmse'] < r['test_rmse']/1.5 else 'OK'))
    print(f'{deg:>8} {r["train_r2"]:>10.4f} {r["test_r2"]:>10.4f} {r["train_rmse"]:>12.4f} {r["test_rmse"]:>12.4f}
{verdict}')
```